In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

In [2]:
DATA_DIR = Path(
    r"C:\Users\ADVAN\UTS-GSD-Rebriane-24031554039"
) / "data" / "processed"

In [3]:
item_map = pd.read_csv(DATA_DIR / "VSM_item_ids_P1.csv")

print(item_map.head())
print(item_map.columns.tolist())

   item_id
0        6
1        9
2       15
3       16
4       25
['item_id']


In [4]:
# Baca matriks dan pemetaan ID Item
vsm_item = sp.load_npz(DATA_DIR / "VSM_item_P1.npz")
item_map = pd.read_csv(DATA_DIR / "VSM_item_ids_P1.csv")

# Periksa nama kolom
print("Kolom pemetaan:", item_map.columns.tolist())
print("Bentuk matriks:", vsm_item.shape)
print(item_map.head())

Kolom pemetaan: ['item_id']
Bentuk matriks: (87308, 185205)
   item_id
0        6
1        9
2       15
3       16
4       25


In [5]:
# GANTI sesuai nama kolom ID Item asli dari output item_map.columns
item_id_col = "item_id"

item_ids = item_map[item_id_col].tolist()

In [6]:
# Pastikan jumlah ID sesuai jumlah baris matriks
assert len(item_ids) == vsm_item.shape[0]

In [7]:
# Pilih satu Item sebagai acuan
reference_idx = 0
reference_id = int(item_ids[reference_idx])

In [8]:
# Hitung kemiripan Item acuan dengan seluruh Item
scores = cosine_similarity(
    vsm_item[reference_idx],
    vsm_item
).ravel()

# Hilangkan Item acuan itu sendiri
scores[reference_idx] = -1

In [9]:
# Ambil 5 Item dengan similarity tertinggi
top_indices = np.argsort(scores)[::-1][:5]

hasil_similarity = pd.DataFrame({
    "item_acuan": [reference_id] * len(top_indices),
    "item_mirip": [int(item_ids[i]) for i in top_indices],
    "similarity": [float(scores[i]) for i in top_indices]
})

print("Item acuan:", reference_id)
print("\nLima Item dengan similarity tertinggi:")
print(hasil_similarity.to_string(index=False))


Item acuan: 6

Lima Item dengan similarity tertinggi:
 item_acuan  item_mirip  similarity
          6      160043    0.447214
          6      236992    0.316228
          6      216965    0.316228
          6      356569    0.258199
          6      413179    0.223607


In [10]:
# Simpan hasil untuk tahap penulisan kembali ke Neo4j
hasil_similarity.to_csv(
    DATA_DIR / "top5_similarity_item_P1.csv",
    index=False
)